# Pydantic Basics: Creating and Using Models
- Notebook by: Adam Lang
- Date: 8-6-2026

## Overview
- Pydantic models are the foundation of data validation in Python. 
- They use type annotations to define the structure and validate data at runtime. This notebook will explore the basic model creation with several examples.

## 1. BaseModel Example

In [ ]:
from pydantic import BaseModel

In [5]:
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int
    city: str

person = Person(name="John Doe", age=30, city="New York")
print(person)

Person(name='John Doe', age=30, city='New York')


In [9]:
## basic pydantic class 
class Person(BaseModel):
    name: str
    age: int
    city: str

person=Person(name="John Doe", age=30, city="New York")
print(person)

name='John Doe' age=30 city='New York'


In [10]:
type(person)

__main__.Person

In [11]:
## create another person
person1=Person(name="Jane Smith", age=25, city=10)
print(person1)

ValidationError: 1 validation error for Person
city
  Input should be a valid string [type=string_type, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

## Summary
- BaseModel allows us to perform automatic data validation. 
- It is very fast because the backend is Rust.

## 2. Model with Optional Fields
- Add optional fields using Python's Optional type:
- Optional indicates the field can be none.
- Notice the type casting does occur below.

In [13]:
from typing import Optional

## create class 
class Employee(BaseModel):
    id: int
    name: str
    department: str 
    salary: Optional[float] = None  # Optional field with default value
    is_active: Optional[bool] = True  # Optional field with default value

In [15]:
# Examples with and without optional fields
emp1 = Employee(id=1, name="Alice", department="HR", salary=50000)
print(emp1) # id=1 name='Alice' department='HR' salary=50000.0 is_active=True

id=1 name='Alice' department='HR' salary=50000.0 is_active=True


### Summary-Definitions
- `Optional[type]:` Indicates the field can be None.
- `Default value (=None or =True)`: Makes the field optional. 
- Required fields must still be provided
- Pydantic validates types even for optional fields when values are provided.

## 3. Lists

In [16]:
### Example with a List 
from pydantic import BaseModel
from typing import List

## create class 
class Classroom(BaseModel):
    room_number: str 
    students: List[str] # List of strings
    capacity: int

In [18]:
## inherit classroom -- create a classroom
classroom = Classroom(
    room_number="A101",
    students=("Alice", "Bob", "Charlie"), ## list of strings (students)
    capacity=30,
)
print(classroom) # room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30

room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


### Note
- If you notice above, I provided the students as a tuple and it type casted it to a LIST because my Pydantic students was preset as `List[str]`

In [19]:
## what happens if we pass a list of integers instead of strings for students?
try:
    invalid_val=Classroom(room_number="A102", students=["Jon",1, 2, 3], capacity=30)
except ValueError as e:
    print(f"Error: {e}")

Error: 3 validation errors for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
students.2
  Input should be a valid string [type=string_type, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
students.3
  Input should be a valid string [type=string_type, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


### Summary
- We can see Pydantic is doing its job type checking above. 

## 4. Model with Nested Models
- Create complex structures with nested models:

In [21]:
## nested models 
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zipcode: int ## changed to int to demonstrate type validation (was str)

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address  # Nested model -- inherits from Address model

# create a customer with nested address
customer = Customer(
    customer_id=1,
    name="Emily",
    address={"street": "123 Main St", "city": "Springfield", "zipcode": "12345"}
    )
print(customer) # customer_id=1 name='Emily' address=Address(street='123 Main St', city='Springfield', zipcode='12345')

customer_id=1 name='Emily' address=Address(street='123 Main St', city='Springfield', zipcode=12345)


## Pydantic Fields: Customization and Constraints
- The Field function in Pydantic enhances model fields beyond basic type hints thus allowing you to specify validation rules, default values, aliases, and more. 
- Here are some comprehensive examples.

In [23]:
from pydantic import BaseModel, Field

## create class with field validation
class Item(BaseModel):
    name:str=Field(min_length=2, max_length=50)
    price:float=Field(gt=0,lt=10000) # price must be greater than 0 and less than 10000
    quantity:int=Field(gt=0, lt=1000) # quantity must be greater than 0 and less than 1000

# Valid instance
item = Item(name="Book", price=19.99, quantity=10)
print(item) # name='Book' price=19.99 quantity=10

name='Book' price=19.99 quantity=10


In [24]:
## another example
from pydantic import BaseModel, Field

## user class
class User(BaseModel):
    username: str=Field(..., description="The unique username of the user") ## note: ... means required field
    age: int=Field(default=18, description="The age of the user, default is 18")
    email: str=Field(default_factory=lambda: "user@example.com", description="The email address of the user")

# Examples
user1 = User(username="alice")
print(user1) # username='alice' age=18 email='user@example.com'

user2 = User(username="bob", age=25, email="bob@domain.com")
print(user2) # username='bob' age=25 email='bob@domain.com'

username='alice' age=18 email='user@example.com'
username='bob' age=25 email='bob@domain.com'


In [29]:
print(User.model_json_schema()) # prints the JSON schema of the User model
print(User.schema_json(indent=2)) # prints the JSON schema of the User model in a pretty format

{'properties': {'username': {'description': 'The unique username of the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'The age of the user, default is 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'The email address of the user', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}
{
  "properties": {
    "username": {
      "description": "The unique username of the user",
      "title": "Username",
      "type": "string"
    },
    "age": {
      "default": 18,
      "description": "The age of the user, default is 18",
      "title": "Age",
      "type": "integer"
    },
    "email": {
      "description": "The email address of the user",
      "title": "Email",
      "type": "string"
    }
  },
  "required": [
    "username"
  ],
  "title": "User",
  "type": "object"
}


/var/folders/y4/gk460f2n31j9j3llmm6lkhv80000gp/T/ipykernel_85227/4138152045.py:2: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(User.schema_json(indent=2)) # prints the JSON schema of the User model in a pretty format
